# CLIFFGUARD — Quantization Safety Evaluation on Google Colab

This notebook runs the CLIFFGUARD pipeline on a free T4 GPU (16 GB VRAM), measuring whether post-training quantization degrades the RLHF-tuned refusal behaviour of instruction-tuned LLMs.

**Pipeline:**
- **Fold A** — calibrate a per-scheme PROBE-RM refusal threshold (Arditi diff-in-means, layer 14, FPR = 5 %)
- **Fold B** — measure three cliff metrics (geometric, Wasserstein, behavioural) vs the FP16 baseline

**Results are synced to Google Drive** after every scheme and survive runtime disconnects. The checkpoint mechanism means you lose at most one scheme of work if the session dies.

> **Note on the kernel restart (cell 4):** Colab ships numpy 2.x; bitsandbytes and Llama-3.2 require numpy < 2. Cell 4 downgrades numpy and self-restarts the kernel once. After the restart, re-run all cells from the top — everything is idempotent and the restart will not fire again.

In [ ]:
!nvidia-smi
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Switch via Runtime > Change runtime type > T4 GPU.")

free, total = torch.cuda.mem_get_info()
print(f"GPU   : {torch.cuda.get_device_name(0)}")
print(f"VRAM  : {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")
print(f"CUDA  : {torch.version.cuda}")

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

for _d in [
    '/content/drive/MyDrive/cliffguard/results',
    '/content/drive/MyDrive/cliffguard/datasets/folds/fold_a',
    '/content/drive/MyDrive/cliffguard/datasets/folds/fold_b',
]:
    os.makedirs(_d, exist_ok=True)

print("Drive mounted — persistent directories ready.")

In [ ]:
%cd /content
!git clone https://github.com/parnish007/CLIFFGUARD.git 2>/dev/null || (cd CLIFFGUARD && git pull)
%cd /content/CLIFFGUARD
!git log -1 --oneline

In [ ]:
import sys, subprocess

# Colab ships numpy 2.x; bitsandbytes + Llama-3.2 require numpy < 2.
# This cell downgrades numpy and restarts the kernel so the old version is
# fully evicted from memory. After restart, re-run from the top — the check
# will pass and no second restart occurs.

try:
    import numpy as _np
    _major = int(_np.__version__.split('.')[0])
except ImportError:
    _major = -1

if _major >= 2:
    print(f"numpy {_np.__version__} detected — downgrading to <2 and restarting.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy<2"], check=True)
    import os, signal
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print(f"numpy {getattr(_np, '__version__', 'not installed')} — OK, continuing.")

In [ ]:
%cd /content/CLIFFGUARD

!pip install -q "numpy<2"
!pip install -q -e /content/CLIFFGUARD
!pip install -q \
    "torch" \
    "transformers>=4.45,<4.50" \
    "bitsandbytes>=0.45" \
    "accelerate" \
    "sentencepiece" \
    "protobuf" \
    "datasets" \
    "huggingface_hub"

import numpy
print(f"numpy {numpy.__version__} — all packages ready.")

In [ ]:
# Pre-built CUDA 12.2 wheel installs in ~5 s. Falls back to a source build
# (~8–10 min) if the wheel is unavailable for this Python/CUDA combination.
!pip install -q llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 \
  || CMAKE_ARGS="-DLLAMA_CUDA=on" pip install --force-reinstall --no-cache-dir -q llama-cpp-python

## HuggingFace Authentication

Llama-3.2 is a gated model — you need to accept the license and provide a read token.

**One-time setup (per HF account):**
1. Accept the model license at [huggingface.co/meta-llama/Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct)
2. Generate a read token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)

**Add it as a Colab Secret (recommended — no copy-paste needed):**
1. Click the **key icon** in the left sidebar
2. Add secret: Name = `cliffguard_test`, Value = your token, toggle **Notebook access ON**

The next cell picks up the secret automatically and falls back to an interactive prompt only if the secret is absent.

In [ ]:
from huggingface_hub import login

_token = None
try:
    from google.colab import userdata
    for _name in ('cliffguard_test', 'HF_TOKEN'):
        try:
            _t = userdata.get(_name)
            if _t:
                _token = _t
                print(f"Authenticated via Colab secret '{_name}'.")
                break
        except Exception:
            continue
except ImportError:
    pass

if _token:
    login(token=_token, add_to_git_credential=False)
else:
    print("No Colab secret found — enter your HF token below.")
    login()

In [ ]:
import sys
sys.path.insert(0, '/content/CLIFFGUARD/notebooks')
import colab_helper as ch

ch.ensure_repo_cwd()
ch.banner()

In [ ]:
ch.ensure_repo_cwd()
!python scripts/dry_run.py --tier A --scheme FP16
!python scripts/dry_run.py --tier C --scheme GGUF_Q3_K_M
print("Smoke test passed.")

In [ ]:
ch.ensure_repo_cwd()
ch.symlink_datasets_from_drive()
!python scripts/download_fold_a.py --download --max 500 --target-dir data/folds/fold_a
!ls -lh data/folds/fold_a/

In [ ]:
ch.ensure_repo_cwd()
config = ch.choose_model()
print(config)

# Override example — force 1B model with two schemes:
# config.update({'model_id': 'meta-llama/Llama-3.2-1B-Instruct',
#                'layer': 8, 'schemes': ['FP16', 'NF4']})

## Fold A — Calibration

For each scheme the pipeline:
1. Loads the model under that quantization scheme
2. Runs Arditi difference-in-means to extract refusal direction **r̂** at layer 14
3. Collects PROBE-RM margin scores over 400 benign prompts
4. Fits threshold **τ_q** at FPR = 5 %
5. Saves artefacts to Drive and advances the checkpoint

The checkpoint survives session disconnects — re-running this cell skips any scheme already marked complete.

> **NF4 loading takes 3–5 minutes on T4. Do not interrupt the cell.**

In [13]:
# Fix: bitsandbytes>=0.45 has CUDA 12.8 support; old <0.45 pin was for CUDA 12.3
!pip install -q "bitsandbytes>=0.45"
import importlib, bitsandbytes
importlib.reload(bitsandbytes)
print(bitsandbytes.__version__)

0.49.2


In [ ]:
# bitsandbytes >= 0.45 ships with CUDA 12.8 support (required on current Colab).
# Earlier versions (< 0.45) target CUDA 12.3 and raise a triton.ops import error.
!pip install -q "bitsandbytes>=0.45"
import importlib, bitsandbytes
importlib.reload(bitsandbytes)
print(bitsandbytes.__version__)

In [ ]:
ch.ensure_repo_cwd()
ch.run_fold_a_with_checkpoint(config)

In [ ]:
ch.ensure_repo_cwd()
ch.sync_artifacts_to_drive()

In [ ]:
ch.ensure_repo_cwd()
ch.assemble_fold_b()

In [ ]:
ch.ensure_repo_cwd()
ch.fold_isolation_audit()

ch.ensure_repo_cwd()
ch.verify_fold_a_complete(config)

In [19]:
# Run this once to pull the fix
!cd /content/CLIFFGUARD && git pull
import importlib, sys
sys.modules.pop('colab_helper', None)
import colab_helper as ch

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 432 bytes | 216.00 KiB/s, done.
From https://github.com/parnish007/CLIFFGUARD
   9f6c6fd..01ce8b1  main       -> origin/main
Updating 9f6c6fd..01ce8b1
Fast-forward
 notebooks/colab_helper.py | 5 ++++-
 1 file changed, 4 insertions(+), 1 deletion(-)


In [ ]:
# Accumulate per-scheme calibration thresholds into calibration_all.json.
# Required when FP16 and NF4 were calibrated in separate sessions, because
# live_execute_fold_a overwrites calibration_summary.json on each call.
import json, pathlib

_run   = pathlib.Path('artifacts/runs/A_colab-4157b7fb14a3_20260520_050923')
_fa    = _run / 'fold_a'
_cal   = json.load(open(_fa / 'calibration_summary.json'))

json.dump(_cal, open(_fa / 'calibration_all.json', 'w'), indent=2)
print("Merged thresholds:", _cal['thresholds'])

_cp = json.load(open(_fa / 'checkpoint.json'))
_cp['completed_schemes'] = [s for s in _cp['completed_schemes'] if s != 'FP16']
_cp['pending_schemes']   = ['FP16']
json.dump(_cp, open(_fa / 'checkpoint.json', 'w'), indent=2)
print("Checkpoint updated — FP16 will recalibrate to rebuild calibration_all.json")

In [ ]:
# Reload helper after pulling calibration-accumulation fix.
!cd /content/CLIFFGUARD && git pull
import sys; sys.modules.pop('colab_helper', None)
import colab_helper as ch

ch.ensure_repo_cwd()
ch.run_fold_b_with_checkpoint(config)
ch.sync_artifacts_to_drive()

In [ ]:
ch.ensure_repo_cwd()
import json, pathlib
from cliffguard.eval.results_writer import list_runs

for run in list_runs(pathlib.Path('artifacts')):
    print('=' * 60)
    print(f'Run: {run.name}')
    for fname in [
        'fold_a/calibration_summary.json',
        'fold_a/checkpoint.json',
        'fold_b/cliff_results.json',
        'fold_b/checkpoint.json',
    ]:
        p = run / fname
        if p.exists():
            print(f'\n--- {fname} ---')
            print(json.dumps(json.load(open(p)), indent=2))

## Results & Next Steps

**Results saved at:** `/content/drive/MyDrive/cliffguard/results/<run_id>/`

**Download locally:**
```python
!zip -r /tmp/cliffguard_run.zip /content/drive/MyDrive/cliffguard/results/
from google.colab import files
files.download('/tmp/cliffguard_run.zip')
```

**Resuming after a disconnect:**
1. Reconnect to a T4 runtime
2. Re-run cells 1–9 (GPU check → Drive mount → clone → numpy pin → install → HF login → helper import)
3. Re-run the choose-model cell to restore `config`
4. Re-run the Fold A cell — checkpoint skips completed schemes
5. Re-run the Fold B cell — picks up from checkpoint

**What to run next to find the safety cliff:**
- Upgrade to Colab Pro (A100) and add `GGUF_Q3_K_M` to `config['schemes']`
- Geometric metric at NF4 was 0.167 (67 % of κ = 0.25); Q3\_K\_M is the predicted cliff zone
- Larger model (Llama-3.1-8B) at the same schemes will test whether cliff location is model-size dependent

**Known limitations:**
- Δ_B-cliff uses PROBE-RM margin proxy, not StrongREJECT + Llama-Guard (Phase C work)
- Fold B corpus is AdvBench + JBB only (HarmBench and ArtPrompt absent)